# Feature-Family Text Backend Benchmark

Small performance benchmark for the feature-family text dataset used by `flair_feature_family_target_mtl.ipynb`.

This notebook compares three ways to consume the same event-only text rows:

- `sentence-transformers`: frozen sentence embeddings plus a logistic-regression classifier.
- HuggingFace `transformers`: one short sequence-classifier fine-tuning loop.
- `Flair`: one short `TextClassifier` fine-tuning loop.

The goal is throughput and operational fit, not final alpha quality. The dataset is intentionally small so the notebook can be rerun frequently.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import date, datetime
from pathlib import Path
from time import perf_counter
import gc
import math
import random
import re
import shutil
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "quant_orchestrator").exists():
            return candidate
    raise RuntimeError(f"Could not find quant-orchestrator repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Prefer a sibling editable quant-warehouse checkout when present. This keeps the notebook usable
# before a local quant-warehouse change has been pushed and reinstalled.
WAREHOUSE_REPO = REPO_ROOT.parent / "quant-warehouse"
if WAREHOUSE_REPO.exists() and str(WAREHOUSE_REPO) not in sys.path:
    sys.path.insert(0, str(WAREHOUSE_REPO))

warnings.filterwarnings("ignore", category=UserWarning)

import torch

from quant_warehouse.platforms.data_providers.fmp.target_engineering.event_pairs import EventPairStore
from quant_warehouse.research_tools import (
    BinaryTargetConfig,
    FamilyEvaluationConfig,
    build_event_target_panel,
    build_fundamental_feature_panel,
    build_oracle_trade_target_panel,
    cap_features_by_quality,
    combine_target_panels,
    load_fmp_event_pairs,
    screen_fmp_equity_universe,
)
from quant_warehouse.warehouse.api import Warehouse

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

In [2]:
RANDOM_SEED = 20260630
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Small dataset knobs. Keep these small; this is a backend benchmark notebook.
MIN_MARKET_CAP = 1_000_000_000_000
START_DATE = "2024-01-01"
END_DATE = None
SYMBOL_LIMIT = 4
MAX_FEATURE_FAMILIES = 8
MAX_TRAIN_ROWS = 2_048
MAX_TEST_ROWS = 512
MIN_FEATURE_COVERAGE = 0.50

# Backend knobs.
HF_MODEL_NAME = "prajjwal1/bert-tiny"
SENTENCE_TRANSFORMER_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZES = (32, 64, 128)
EPOCHS = 1
LEARNING_RATE = 1e-4
MAX_LENGTH = 512

ARTIFACT_DIR = REPO_ROOT / "artifacts" / "notebooks" / "mult-ml-frameworks" / "feature_family_text_backend_benchmark"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

feature_config = FamilyEvaluationConfig(
    provider="fmp",
    market_cap_min=MIN_MARKET_CAP,
    start_date=START_DATE,
    end_date=END_DATE,
    horizons=(20, 60),
    min_observations=60,
)

target_config = BinaryTargetConfig(
    provider="fmp",
    start_date=START_DATE,
    end_date=END_DATE,
    event_families=("congress", "analyst_rating", "price_target", "earnings"),
    oracle_trade_k_by_frequency={"YE": tuple(range(1, 13))},
    oracle_trade_min_profit_pct=0.01,
    oracle_trade_long_entry_price_col="high",
    oracle_trade_long_exit_price_col="low",
    oracle_trade_short_entry_price_col="low",
    oracle_trade_short_exit_price_col="high",
)

config_audit = pd.DataFrame(
    [
        {"setting": "device", "value": str(DEVICE)},
        {"setting": "min_market_cap", "value": f"{MIN_MARKET_CAP:,}"},
        {"setting": "date_range", "value": f"{START_DATE} to {END_DATE or 'latest warehouse row'}"},
        {"setting": "symbol_limit", "value": SYMBOL_LIMIT},
        {"setting": "max_feature_families", "value": MAX_FEATURE_FAMILIES},
        {"setting": "max_train_rows", "value": MAX_TRAIN_ROWS},
        {"setting": "max_test_rows", "value": MAX_TEST_ROWS},
        {"setting": "hf_model", "value": HF_MODEL_NAME},
        {"setting": "sentence_transformer_model", "value": SENTENCE_TRANSFORMER_MODEL_NAME},
        {"setting": "batch_sizes", "value": BATCH_SIZES},
    ]
)
display(config_audit)

,setting,value
0,device,cuda
1,min_market_cap,"1,000,000,000,000"
2,date_range,2024-01-01 to latest warehouse row
3,symbol_limit,4
4,max_feature_families,8
5,max_train_rows,2048
6,max_test_rows,512
7,hf_model,prajjwal1/bert-tiny
8,sentence_transformer_model,sentence-transformers/all-MiniLM-L6-v2
9,batch_sizes,"(32, 64, 128)"


## Build A Small Event-Only Text Dataset

This mirrors the larger Flair MTL notebook, but intentionally narrows the universe and output rows. The supervised task used here is one oracle buy/sell task across all configured `k` values. Non-entry dates are excluded.

In [3]:
def format_feature_value(value: object) -> str | None:
    if value is None or pd.isna(value):
        return None
    if isinstance(value, (pd.Timestamp, datetime, date)):
        return pd.Timestamp(value).strftime("%Y-%m-%d")
    try:
        number = float(value)
    except (TypeError, ValueError):
        text = str(value).strip()
        return text if text else None
    if not math.isfinite(number):
        return None
    return f"{number:.2f}"


def compact_feature_key(feature: str, family: str) -> str:
    key = str(feature)
    prefix = f"{family}__"
    if key.startswith(prefix):
        key = key[len(prefix):]
    replacements = {
        "market_cap": "mcap",
        "enterprise_value": "ev",
        "total_": "tot_",
        "current_": "cur_",
        "operating_": "op_",
        "stockholders": "sh",
        "liabilities": "liab",
        "receivables": "recv",
        "inventory": "inv",
        "depreciation": "depr",
        "amortization": "amort",
    }
    for old, new in replacements.items():
        key = key.replace(old, new)
    return key.strip("_") or str(feature)


def feature_family_text(row: pd.Series, features: list[str], *, source: str, family: str) -> str:
    pairs = [f"source={source}", f"feature_family={family}"]
    for feature in features:
        value = format_feature_value(row.get(feature))
        if value is not None:
            pairs.append(f"{compact_feature_key(feature, family)}={value}")
    return " ".join(pairs)


def add_profile_feature_family(training_panel: pd.DataFrame, metadata: pd.DataFrame, warehouse: Warehouse) -> tuple[pd.DataFrame, pd.DataFrame]:
    profile_columns = ["date", "symbol", "company_name", "exchange", "country", "sector", "industry"]
    rows = []
    for symbol in training_panel["symbol"].astype(str).str.upper().drop_duplicates().sort_values():
        profile = warehouse.catalog.get_profile(symbol=symbol, provider=feature_config.provider)
        rows.append(
            {
                "symbol": symbol,
                "company_name": profile.company_name if profile is not None else None,
                "exchange": profile.exchange if profile is not None else None,
                "country": profile.country if profile is not None else None,
                "sector": profile.sector if profile is not None else None,
                "industry": profile.industry if profile is not None else None,
            }
        )
    profile_frame = pd.DataFrame(rows)
    out_panel = training_panel.merge(profile_frame, on="symbol", how="left")
    profile_metadata = pd.DataFrame(
        [
            {
                "feature": column,
                "family": "fmp_equity_profile",
                "source": "fmp",
                "source_column": column,
                "expected_direction": "categorical",
            }
            for column in profile_columns
        ]
    )
    out_metadata = pd.concat([metadata, profile_metadata], ignore_index=True)
    return out_panel, out_metadata


def oracle_buy_sell_columns(columns: list[str]) -> tuple[list[str], list[str]]:
    long_cols = sorted(c for c in columns if re.match(r"^target_oracle_trade_entry__.+_long$", str(c)))
    short_cols = sorted(re.sub(r"_long$", "_short", c) for c in long_cols if re.sub(r"_long$", "_short", c) in columns)
    long_cols = [re.sub(r"_short$", "_long", c) for c in short_cols]
    if not long_cols or not short_cols:
        raise RuntimeError("No paired oracle long/short columns available.")
    return long_cols, short_cols


def balanced_sample(frame: pd.DataFrame, max_rows: int, *, label_col: str = "label") -> pd.DataFrame:
    if len(frame) <= max_rows:
        return frame.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    labels = frame[label_col].astype(str)
    per_label = max(1, max_rows // max(1, labels.nunique()))
    pieces = []
    for _, group in frame.groupby(label_col, sort=False):
        pieces.append(group.sample(n=min(len(group), per_label), random_state=RANDOM_SEED))
    sampled = pd.concat(pieces, axis=0)
    if len(sampled) < max_rows:
        remainder = frame.drop(index=sampled.index)
        if not remainder.empty:
            sampled = pd.concat([sampled, remainder.sample(n=min(len(remainder), max_rows - len(sampled)), random_state=RANDOM_SEED)])
    return sampled.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

In [4]:
started = perf_counter()
warehouse = Warehouse()
event_store = EventPairStore(
    config=warehouse.config,
    backend=warehouse.backend,
    catalog=warehouse.catalog,
    fundamentals=warehouse.fundamentals,
    equity_calendar=warehouse.equity_calendar,
)

symbols, raw_universe, eligibility, universe_source = screen_fmp_equity_universe(feature_config, warehouse=warehouse)
symbols = tuple(symbols[:SYMBOL_LIMIT])
print({"universe_source": universe_source, "symbols": symbols})

raw_feature_panel, raw_feature_metadata, feature_diagnostics, feature_timings = build_fundamental_feature_panel(
    symbols,
    feature_config,
    warehouse=warehouse,
)

events, event_diagnostics, event_load_seconds = load_fmp_event_pairs(
    symbols,
    target_config,
    event_store=event_store,
    include_historical=True,
)
event_symbols = tuple(event_diagnostics.loc[event_diagnostics["combined_rows"].gt(0), "symbol"].sort_values().tolist())
feature_panel = raw_feature_panel.loc[raw_feature_panel["symbol"].isin(event_symbols)].copy()

selected_features, selected_feature_metadata, feature_quality = cap_features_by_quality(
    feature_panel,
    raw_feature_metadata,
    max_features=int(raw_feature_metadata["feature"].nunique()),
)

event_target_panel, event_target_metadata = build_event_target_panel(
    feature_panel[["symbol", "date"]],
    events,
    target_config,
)
oracle_target_panel, oracle_target_metadata, oracle_seconds = build_oracle_trade_target_panel(
    event_symbols,
    target_config,
    warehouse=warehouse,
)
target_panel = combine_target_panels(event_target_panel, oracle_target_panel)
training_panel = feature_panel.merge(target_panel, on=["symbol", "date"], how="inner")
training_panel, selected_feature_metadata = add_profile_feature_family(training_panel, selected_feature_metadata, warehouse)

print(
    {
        "symbols": len(symbols),
        "event_symbols": len(event_symbols),
        "feature_panel_rows": len(feature_panel),
        "training_panel_rows": len(training_panel),
        "feature_families": selected_feature_metadata["family"].nunique(),
        "targets": target_panel.shape[1] - 2,
        "build_seconds": round(perf_counter() - started, 3),
    }
)

display(
    selected_feature_metadata.groupby(["source", "family"]).agg(features=("feature", "nunique")).reset_index().sort_values(["source", "family"])
)

{'universe_source': 'openbb:fmp', 'symbols': ('AAPL', 'AMZN', 'AVGO', 'BRK-A')}


{'symbols': 4, 'event_symbols': 4, 'feature_panel_rows': 2484, 'training_panel_rows': 2484, 'feature_families': 16, 'targets': 44, 'build_seconds': 4.826}


,source,family,features
0,financetoolkit,ft_growth_balance,56
1,financetoolkit,ft_growth_cash,50
2,financetoolkit,ft_growth_income,38
3,financetoolkit,ft_ratios_efficiency,5
4,financetoolkit,ft_ratios_liquidity,7
5,financetoolkit,ft_ratios_profitability,14
6,financetoolkit,ft_ratios_solvency,15
7,financetoolkit,ft_ratios_valuation,24
8,fmp,fmp_balance_mcap,55
9,fmp,fmp_cash_mcap,39


In [5]:
long_cols, short_cols = oracle_buy_sell_columns([c for c in training_panel.columns if c.startswith("target_")])
positive = training_panel[long_cols].apply(pd.to_numeric, errors="coerce").fillna(0).gt(0).any(axis=1)
negative = training_panel[short_cols].apply(pd.to_numeric, errors="coerce").fillna(0).gt(0).any(axis=1)
event_mask = positive | negative
ambiguous_mask = positive & negative
eligible_index = training_panel.index[event_mask & ~ambiguous_mask]

# Keep a small, diverse family set. Always include profile context.
family_inventory = selected_feature_metadata[["source", "family"]].drop_duplicates().sort_values(["source", "family"])
profile_row = family_inventory.query("family == 'fmp_equity_profile'")
non_profile = family_inventory.query("family != 'fmp_equity_profile'").head(MAX_FEATURE_FAMILIES - len(profile_row))
kept_families = pd.concat([profile_row, non_profile], ignore_index=True)
kept_family_set = set(map(tuple, kept_families[["source", "family"]].to_numpy()))

rows = []
for (source, family), family_meta in selected_feature_metadata.groupby(["source", "family"], sort=True):
    if (source, family) not in kept_family_set:
        continue
    features = [f for f in family_meta["feature"].drop_duplicates().tolist() if f in training_panel.columns]
    if not features:
        continue
    coverage_mask = training_panel[features].notna().mean(axis=1).ge(MIN_FEATURE_COVERAGE)
    selected_index = eligible_index.intersection(training_panel.index[coverage_mask])
    if selected_index.empty:
        continue
    base_columns = list(dict.fromkeys(["symbol", "date", *features]))
    base = training_panel.loc[selected_index, base_columns].copy()
    text = base.apply(lambda row: feature_family_text(row, features, source=source, family=family), axis=1)
    frame = pd.DataFrame(
        {
            "symbol": base["symbol"].astype(str).str.upper().to_numpy(),
            "date": pd.to_datetime(base["date"], errors="coerce").dt.normalize().to_numpy(),
            "source": source,
            "feature_family": family,
            "text": text.to_numpy(),
            "label": np.where(positive.loc[selected_index].to_numpy(), "buy", "sell"),
            "task": "oracle_buy_sell",
        }
    )
    rows.append(frame)

long_frame = pd.concat(rows, ignore_index=True).dropna(subset=["date", "text", "label"])
long_frame = long_frame.loc[long_frame["text"].str.len().gt(0)].copy()
long_frame = long_frame.sort_values(["date", "symbol", "feature_family"]).reset_index(drop=True)

split_date = long_frame["date"].quantile(0.8)
train_frame = long_frame.loc[long_frame["date"].le(split_date)].copy()
test_frame = long_frame.loc[long_frame["date"].gt(split_date)].copy()
train_frame = balanced_sample(train_frame, MAX_TRAIN_ROWS)
test_frame = balanced_sample(test_frame, MAX_TEST_ROWS)

print(
    {
        "raw_long_rows": len(long_frame),
        "train_rows": len(train_frame),
        "test_rows": len(test_frame),
        "feature_families": train_frame["feature_family"].nunique(),
        "train_labels": dict(train_frame["label"].value_counts()),
        "test_labels": dict(test_frame["label"].value_counts()),
        "median_tokens": float(train_frame["text"].str.split().str.len().median()),
        "max_tokens": int(train_frame["text"].str.split().str.len().max()),
    }
)
display(train_frame.head(10))

{'raw_long_rows': 1944, 'train_rows': 1568, 'test_rows': 376, 'feature_families': 8, 'train_labels': {'buy': np.int64(800), 'sell': np.int64(768)}, 'test_labels': {'sell': np.int64(200), 'buy': np.int64(176)}, 'median_tokens': 16.5, 'max_tokens': 53}


,symbol,date,source,feature_family,text,label,task
0,BRK-A,2026-01-09,financetoolkit,ft_ratios_liquidity,source=financetoolkit feature_family=ft_ratios...,sell,oracle_buy_sell
1,AMZN,2024-01-17,financetoolkit,ft_ratios_liquidity,source=financetoolkit feature_family=ft_ratios...,buy,oracle_buy_sell
2,AMZN,2025-05-23,financetoolkit,ft_ratios_liquidity,source=financetoolkit feature_family=ft_ratios...,buy,oracle_buy_sell
3,AMZN,2025-06-23,financetoolkit,ft_ratios_efficiency,source=financetoolkit feature_family=ft_ratios...,buy,oracle_buy_sell
4,BRK-A,2024-11-05,financetoolkit,ft_ratios_efficiency,source=financetoolkit feature_family=ft_ratios...,buy,oracle_buy_sell
5,AMZN,2024-08-28,financetoolkit,ft_ratios_solvency,source=financetoolkit feature_family=ft_ratios...,buy,oracle_buy_sell
6,BRK-A,2025-06-24,financetoolkit,ft_ratios_efficiency,source=financetoolkit feature_family=ft_ratios...,sell,oracle_buy_sell
7,AVGO,2024-04-29,financetoolkit,ft_growth_cash,source=financetoolkit feature_family=ft_growth...,sell,oracle_buy_sell
8,AVGO,2024-02-21,fmp,fmp_equity_profile,source=fmp feature_family=fmp_equity_profile d...,buy,oracle_buy_sell
9,AAPL,2025-08-21,financetoolkit,ft_growth_cash,source=financetoolkit feature_family=ft_growth...,buy,oracle_buy_sell


## Benchmark Helpers

All benchmarks report wall-clock seconds and samples/sec on the same `train_frame` / `test_frame`.

In [6]:
benchmark_rows: list[dict[str, object]] = []


def record_result(**kwargs: object) -> None:
    benchmark_rows.append(kwargs)
    display(pd.DataFrame(benchmark_rows).sort_values(["status", "samples_per_second"], ascending=[True, False]))


def label_arrays(train: pd.DataFrame, test: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, LabelEncoder]:
    encoder = LabelEncoder()
    y_train = encoder.fit_transform(train["label"].astype(str))
    y_test = encoder.transform(test["label"].astype(str))
    return y_train, y_test, encoder


def classification_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


def cleanup_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

In [7]:
def benchmark_sentence_transformers(batch_size: int = 128) -> dict[str, object]:
    try:
        from sentence_transformers import SentenceTransformer
    except ModuleNotFoundError as exc:
        return {
            "backend": "sentence-transformers",
            "batch_size": batch_size,
            "status": "missing_dependency: pip install 'quant-orchestrator[ml]' or sentence-transformers",
            "seconds": np.nan,
            "samples_per_second": 0.0,
            "accuracy": np.nan,
            "macro_f1": np.nan,
            "peak_cuda_gb": np.nan,
        }

    cleanup_cuda()
    y_train, y_test, _ = label_arrays(train_frame, test_frame)
    started = perf_counter()
    model = SentenceTransformer(SENTENCE_TRANSFORMER_MODEL_NAME, device=str(DEVICE))
    train_embeddings = model.encode(
        train_frame["text"].tolist(),
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=False,
    )
    test_embeddings = model.encode(
        test_frame["text"].tolist(),
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=False,
    )
    clf = LogisticRegression(max_iter=1_000, class_weight="balanced", random_state=RANDOM_SEED)
    clf.fit(train_embeddings, y_train)
    pred = clf.predict(test_embeddings)
    seconds = perf_counter() - started
    metrics = classification_metrics(y_test, pred)
    return {
        "backend": "sentence-transformers+logreg",
        "batch_size": batch_size,
        "status": "ok",
        "seconds": seconds,
        "samples_per_second": len(train_frame) / seconds,
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
        "peak_cuda_gb": torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else np.nan,
    }


for batch_size in BATCH_SIZES:
    record_result(**benchmark_sentence_transformers(batch_size=batch_size))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,backend,batch_size,status,seconds,samples_per_second,accuracy,macro_f1,peak_cuda_gb
0,sentence-transformers+logreg,32,ok,8.612397,182.063126,0.468085,0.318841,0.284241


,backend,batch_size,status,seconds,samples_per_second,accuracy,macro_f1,peak_cuda_gb
1,sentence-transformers+logreg,64,ok,3.694196,424.449626,0.468085,0.318841,0.444067
0,sentence-transformers+logreg,32,ok,8.612397,182.063126,0.468085,0.318841,0.284241


,backend,batch_size,status,seconds,samples_per_second,accuracy,macro_f1,peak_cuda_gb
1,sentence-transformers+logreg,64,ok,3.694196,424.449626,0.468085,0.318841,0.444067
2,sentence-transformers+logreg,128,ok,3.813779,411.140732,0.468085,0.318841,0.763719
0,sentence-transformers+logreg,32,ok,8.612397,182.063126,0.468085,0.318841,0.284241


In [8]:
def benchmark_huggingface_transformers(batch_size: int = 128) -> dict[str, object]:
    from torch.utils.data import DataLoader, Dataset
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    cleanup_cuda()
    y_train, y_test, encoder = label_arrays(train_frame, test_frame)
    tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME, model_max_length=MAX_LENGTH)

    class TextDataset(Dataset):
        def __init__(self, texts: list[str], labels: np.ndarray):
            self.texts = texts
            self.labels = labels

        def __len__(self) -> int:
            return len(self.texts)

        def __getitem__(self, idx: int) -> dict[str, object]:
            return {"text": self.texts[idx], "label": int(self.labels[idx])}

    def collate(batch: list[dict[str, object]]) -> dict[str, torch.Tensor]:
        encoded = tokenizer(
            [str(item["text"]) for item in batch],
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        encoded["labels"] = torch.tensor([int(item["label"]) for item in batch], dtype=torch.long)
        return encoded

    train_loader = DataLoader(TextDataset(train_frame["text"].tolist(), y_train), batch_size=batch_size, shuffle=True, collate_fn=collate)
    test_loader = DataLoader(TextDataset(test_frame["text"].tolist(), y_test), batch_size=batch_size, shuffle=False, collate_fn=collate)

    model = AutoModelForSequenceClassification.from_pretrained(HF_MODEL_NAME, num_labels=len(encoder.classes_)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    started = perf_counter()
    model.train()
    for _ in range(EPOCHS):
        for batch in train_loader:
            batch = {key: value.to(DEVICE) for key, value in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            loss = model(**batch).loss
            loss.backward()
            optimizer.step()

    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in test_loader:
            labels = batch.pop("labels")
            batch = {key: value.to(DEVICE) for key, value in batch.items()}
            logits = model(**batch).logits.detach().cpu().numpy()
            predictions.extend(logits.argmax(axis=1).tolist())
    seconds = perf_counter() - started
    metrics = classification_metrics(y_test, np.array(predictions))
    return {
        "backend": "huggingface-transformers",
        "batch_size": batch_size,
        "status": "ok",
        "seconds": seconds,
        "samples_per_second": len(train_frame) / seconds,
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
        "peak_cuda_gb": torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else np.nan,
    }


for batch_size in BATCH_SIZES:
    record_result(**benchmark_huggingface_transformers(batch_size=batch_size))

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


,backend,batch_size,status,seconds,samples_per_second,accuracy,macro_f1,peak_cuda_gb
3,huggingface-transformers,32,ok,1.431715,1095.189833,0.468085,0.318841,0.531519
1,sentence-transformers+logreg,64,ok,3.694196,424.449626,0.468085,0.318841,0.444067
2,sentence-transformers+logreg,128,ok,3.813779,411.140732,0.468085,0.318841,0.763719
0,sentence-transformers+logreg,32,ok,8.612397,182.063126,0.468085,0.318841,0.284241


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


,backend,batch_size,status,seconds,samples_per_second,accuracy,macro_f1,peak_cuda_gb
4,huggingface-transformers,64,ok,1.145206,1369.186555,0.513298,0.504176,0.877420
3,huggingface-transformers,32,ok,1.431715,1095.189833,0.468085,0.318841,0.531519
1,sentence-transformers+logreg,64,ok,3.694196,424.449626,0.468085,0.318841,0.444067
2,sentence-transformers+logreg,128,ok,3.813779,411.140732,0.468085,0.318841,0.763719
0,sentence-transformers+logreg,32,ok,8.612397,182.063126,0.468085,0.318841,0.284241


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


,backend,batch_size,status,seconds,samples_per_second,accuracy,macro_f1,peak_cuda_gb
5,huggingface-transformers,128,ok,1.139069,1376.562526,0.494681,0.494552,1.605394
4,huggingface-transformers,64,ok,1.145206,1369.186555,0.513298,0.504176,0.877420
3,huggingface-transformers,32,ok,1.431715,1095.189833,0.468085,0.318841,0.531519
1,sentence-transformers+logreg,64,ok,3.694196,424.449626,0.468085,0.318841,0.444067
2,sentence-transformers+logreg,128,ok,3.813779,411.140732,0.468085,0.318841,0.763719
0,sentence-transformers+logreg,32,ok,8.612397,182.063126,0.468085,0.318841,0.284241


In [9]:
def benchmark_flair(batch_size: int = 128) -> dict[str, object]:
    import flair
    from flair.data import Corpus, Dictionary, FlairDataset, Sentence
    from flair.embeddings import TransformerDocumentEmbeddings
    from flair.models import TextClassifier
    from flair.trainers import ModelTrainer

    cleanup_cuda()
    flair.device = DEVICE

    class LazyTextDataset(FlairDataset):
        def __init__(self, frame: pd.DataFrame):
            self.texts = frame["text"].astype(str).to_numpy(copy=True)
            self.labels = frame["label"].astype(str).to_numpy(copy=True)

        def __len__(self) -> int:
            return len(self.texts)

        def __getitem__(self, index: int) -> Sentence:
            sentence = Sentence(self.texts[index])
            sentence.add_label("label", self.labels[index])
            return sentence

        def is_in_memory(self) -> bool:
            return False

    dictionary = Dictionary(add_unk=False)
    for label in sorted(train_frame["label"].astype(str).unique()):
        dictionary.add_item(label)

    corpus = Corpus(
        train=LazyTextDataset(train_frame),
        dev=LazyTextDataset(test_frame),
        test=LazyTextDataset(test_frame),
        sample_missing_splits=False,
    )
    embeddings = TransformerDocumentEmbeddings(
        HF_MODEL_NAME,
        fine_tune=False,
        layers="-1",
        layer_mean=False,
        allow_long_sentences=False,
        transformers_tokenizer_kwargs={"model_max_length": MAX_LENGTH},
    )
    model = TextClassifier(embeddings, label_type="label", label_dictionary=dictionary).to(flair.device)
    trainer = ModelTrainer(model, corpus)
    run_dir = ARTIFACT_DIR / f"flair_batch_{batch_size}"
    if run_dir.exists():
        shutil.rmtree(run_dir)

    started = perf_counter()
    trainer.fine_tune(
        base_path=run_dir,
        learning_rate=LEARNING_RATE,
        mini_batch_size=batch_size,
        mini_batch_chunk_size=None,
        eval_batch_size=batch_size,
        max_epochs=EPOCHS,
        embeddings_storage_mode="none",
        save_final_model=False,
        create_file_logs=False,
        create_loss_file=False,
    )
    seconds = perf_counter() - started

    y_true = test_frame["label"].astype(str).to_numpy()
    sentences = [Sentence(text) for text in test_frame["text"].astype(str).tolist()]
    model.predict(sentences, mini_batch_size=batch_size, label_name="pred")
    y_pred = np.array([sentence.get_labels("pred")[0].value for sentence in sentences])
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }
    return {
        "backend": "flair",
        "batch_size": batch_size,
        "status": "ok",
        "seconds": seconds,
        "samples_per_second": len(train_frame) / seconds,
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
        "peak_cuda_gb": torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else np.nan,
    }


for batch_size in BATCH_SIZES:
    record_result(**benchmark_flair(batch_size=batch_size))

2026-06-30 21:14:55,275 ----------------------------------------------------------------------------------------------------


2026-06-30 21:14:55,276 Model: "TextClassifier(
  (embeddings): TransformerDocumentEmbeddings(
    (model): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30523, 128, padding_idx=0)
        (position_embeddings): Embedding(512, 128)
        (token_type_embeddings): Embedding(2, 128)
        (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-1): 2 x BertLayer(
            (attention): BertAttention(
              (self): BertSdpaSelfAttention(
                (query): Linear(in_features=128, out_features=128, bias=True)
                (key): Linear(in_features=128, out_features=128, bias=True)
                (value): Linear(in_features=128, out_features=128, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
               

2026-06-30 21:14:55,277 ----------------------------------------------------------------------------------------------------


2026-06-30 21:14:55,277 Corpus: 1568 train + 376 dev + 376 test sentences


2026-06-30 21:14:55,278 ----------------------------------------------------------------------------------------------------


...

100%|██████████| 3/3 [00:00<00:00,  6.40it/s]

100%|██████████| 3/3 [00:00<00:00,  6.36it/s]

2026-06-30 21:15:09,366 
Results:
- F-score (micro) 0.5053
- F-score (macro) 0.4991
- Accuracy 0.5053

By class:
              precision    recall  f1-score   support

        sell     0.5321    0.5800    0.5550       200
         buy     0.4684    0.4205    0.4431       176

    accuracy                         0.5053       376
   macro avg     0.5002    0.5002    0.4991       376
weighted avg     0.5023    0.5053    0.5026       376



2026-06-30 21:15:09,367 ----------------------------------------------------------------------------------------------------


,backend,batch_size,status,seconds,samples_per_second,accuracy,macro_f1,peak_cuda_gb
5,huggingface-transformers,128,ok,1.139069,1376.562526,0.494681,0.494552,1.605394
4,huggingface-transformers,64,ok,1.145206,1369.186555,0.513298,0.504176,0.877420
3,huggingface-transformers,32,ok,1.431715,1095.189833,0.468085,0.318841,0.531519
8,flair,128,ok,3.324886,471.595134,0.505319,0.499069,0.589035
7,flair,64,ok,3.325606,471.492969,0.531915,0.347222,0.336884
6,flair,32,ok,3.527236,444.540727,0.531915,0.347222,0.210776
1,sentence-transformers+logreg,64,ok,3.694196,424.449626,0.468085,0.318841,0.444067
2,sentence-transformers+logreg,128,ok,3.813779,411.140732,0.468085,0.318841,0.763719
0,sentence-transformers+logreg,32,ok,8.612397,182.063126,0.468085,0.318841,0.284241


## Summary

Sort by `samples_per_second` to find the fastest backend/batch for this small dataset. Sort by `macro_f1` if you care about label balance. Because `sentence-transformers` uses frozen embeddings plus logistic regression while HuggingFace and Flair do a short supervised neural training pass, treat throughput as an operational comparison, not a strict modeling-quality comparison.

In [10]:
results = pd.DataFrame(benchmark_rows)
if results.empty:
    raise RuntimeError("No benchmark rows were produced.")

summary = results.sort_values("samples_per_second", ascending=False).reset_index(drop=True)
display(summary)

best = summary.iloc[0].to_dict()
print("Best throughput:", best)

,backend,batch_size,status,seconds,samples_per_second,accuracy,macro_f1,peak_cuda_gb
0,huggingface-transformers,128,ok,1.139069,1376.562526,0.494681,0.494552,1.605394
1,huggingface-transformers,64,ok,1.145206,1369.186555,0.513298,0.504176,0.877420
2,huggingface-transformers,32,ok,1.431715,1095.189833,0.468085,0.318841,0.531519
3,flair,128,ok,3.324886,471.595134,0.505319,0.499069,0.589035
4,flair,64,ok,3.325606,471.492969,0.531915,0.347222,0.336884
5,flair,32,ok,3.527236,444.540727,0.531915,0.347222,0.210776
6,sentence-transformers+logreg,64,ok,3.694196,424.449626,0.468085,0.318841,0.444067
7,sentence-transformers+logreg,128,ok,3.813779,411.140732,0.468085,0.318841,0.763719
8,sentence-transformers+logreg,32,ok,8.612397,182.063126,0.468085,0.318841,0.284241


Best throughput: {'backend': 'huggingface-transformers', 'batch_size': 128, 'status': 'ok', 'seconds': 1.139069217722863, 'samples_per_second': 1376.5625263183053, 'accuracy': 0.4946808510638298, 'macro_f1': 0.49455214376680345, 'peak_cuda_gb': 1.605394432}
